In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install imagecorruptions

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Subset
from imagecorruptions import corrupt, get_corruption_names
import numpy as np
import random
from PIL import Image
import os
import gc
import pandas as pd
import time
from tqdm import tqdm 
from torch.amp import autocast, GradScaler 

# Setup directory for results
os.makedirs('results/plots', exist_ok=True)
CSV_PATH = 'results/experiment_accuracies.csv'

# Initialize CSV if it does not exist
if not os.path.exists(CSV_PATH):
    with open(CSV_PATH, 'w') as f:
        f.write('Dataset,Model,Experiment,Test_Accuracy,Corrupted_Test_Accuracy\n')

# Configuration constants
EPOCHS = 10
BATCH_SIZE = 128
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMAGENET_PATH = '/kaggle/input/imagenet100' # Path for ImageNet-100 dataset

# All 15 standard corruptions from the library at severity 2 as per rubric
ALL_CORRUPTIONS = get_corruption_names()

# Custom transform class to apply imagecorruptions library
class CorruptionTransform:
    def __init__(self, corruption_names, severity=2):
        self.corruption_names = corruption_names
        self.severity = severity

    def __call__(self, img):
        # Convert PIL to numpy for the corruption library
        img_np = np.array(img.convert("RGB"))
        # Pick a random corruption from the provided list
        c_name = random.choice(self.corruption_names)
        corrupted = corrupt(img_np, corruption_name=c_name, severity=self.severity)
        # Convert back to PIL
        return Image.fromarray(corrupted)

def get_dataloaders(dataset_name, val_mode='clean'):
    """
    Handles data loading and 20% validation split as required by the assignment.
    """
    is_im100 = (dataset_name == 'imagenet100')
    is_fmnist = (dataset_name == 'fmnist')
    
    # Standard training transforms
    train_t = [transforms.Resize((224, 224)) if is_im100 else transforms.Resize((32, 32))]
    if is_fmnist: train_t.append(transforms.Grayscale(3))
    train_t.extend([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    
    # Base evaluation transforms
    eval_t = [transforms.Resize((224, 224)) if is_im100 else transforms.Resize((32, 32))]
    if is_fmnist: eval_t.append(transforms.Grayscale(3))
    
    # Determine corruption for validation/test sets based on experiment mode
    if val_mode == 'corrupted_all':
        eval_t.insert(1, CorruptionTransform(ALL_CORRUPTIONS, severity=2))
    elif val_mode == 'corrupted_optimized':
        # Optimized subset (e.g., 5 representative corruptions)
        optimized = ['gaussian_noise', 'motion_blur', 'pixelate', 'jpeg_compression', 'snow']
        eval_t.insert(1, CorruptionTransform(optimized, severity=2))

    eval_t.extend([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

    # Load Raw Datasets
    if dataset_name == 'cifar10':
        full_train = torchvision.datasets.CIFAR10('./data', train=True, download=True)
        test_ds = torchvision.datasets.CIFAR10('./data', train=False, download=True)
    elif dataset_name == 'fmnist':
        full_train = torchvision.datasets.FashionMNIST('./data', train=True, download=True)
        test_ds = torchvision.datasets.FashionMNIST('./data', train=False, download=True)
    else:
        full_train = torchvision.datasets.ImageFolder(os.path.join(IMAGENET_PATH, 'train'))
        test_ds = torchvision.datasets.ImageFolder(os.path.join(IMAGENET_PATH, 'val'))

    # Strict 20% Validation Split Requirement
    num_train = len(full_train)
    indices = list(range(num_train))
    split = int(0.2 * num_train) # 20% for validation
    
    # Fix seed for split consistency across experiments
    np.random.seed(42)
    np.random.shuffle(indices)
    
    train_idx, val_idx = indices[split:], indices[:split]

    # Apply specific transforms to subsets
    train_ds = Subset(full_train, train_idx)
    train_ds.dataset.transform = transforms.Compose(train_t)
    
    val_ds = Subset(full_train, val_idx)
    val_ds.dataset.transform = transforms.Compose(eval_t)
    
    test_ds.transform = transforms.Compose(eval_t)

    # Dataloaders
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    return train_loader, val_loader, test_loader

def get_model(name, num_classes):
    """
    Initializes models: VGG, ResNet, ConvNeXT, and ViT.
    """
    if name == 'resnet18':
        m = models.resnet18(weights='DEFAULT')
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == 'vgg16':
        m = models.vgg16_bn(weights='DEFAULT')
        m.classifier[6] = nn.Linear(4096, num_classes)
    elif name == 'convnext_tiny':
        m = models.convnext_tiny(weights='DEFAULT')
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, num_classes)
    elif name == 'vit_b_16':
        m = models.vit_b_16(weights='DEFAULT')
        m.heads.head = nn.Linear(m.heads.head.in_features, num_classes)
    return m.to(DEVICE)

def train_and_eval(dataset_name, model_name, exp_name, val_mode):
    # Determine classes
    num_classes = 100 if dataset_name == 'imagenet100' else 10
    
    # Load data for this specific experiment
    train_loader, val_loader, test_loader = get_dataloaders(dataset_name, val_mode)
    
    # Initialize model, optimizer and scaler for mixed precision
    model = get_model(model_name, num_classes)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)
    scaler = GradScaler('cuda')
    criterion = nn.CrossEntropyLoss()

    # Training Loop
    for epoch in range(EPOCHS):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            # Auto-resize if model expects 224x224 (ViT/ConvNeXt)
            if model_name in ['vit_b_16', 'convnext_tiny'] and images.shape[-1] < 224:
                images = F.interpolate(images, size=(224, 224), mode='bilinear')

            optimizer.zero_grad()
            with autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

    # Final Evaluation on Test Set
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if model_name in ['vit_b_16', 'convnext_tiny'] and images.shape[-1] < 224:
                images = F.interpolate(images, size=(224, 224), mode='bilinear')
            
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    
    # Log results to CSV
    with open(CSV_PATH, 'a') as f:
        f.write(f"{dataset_name},{model_name},{exp_name},{accuracy:.2f}\n")
    
    # Memory management
    del model, optimizer; gc.collect(); torch.cuda.empty_cache()

# Main Execution Loop per Assignment Requirements
datasets = ['cifar10', 'fmnist', 'imagenet100']
models_to_train = ['vgg16', 'resnet18', 'convnext_tiny', 'vit_b_16']

for ds in datasets:
    for model_n in models_to_train:
        # Experiment 1: Clean Validation
        train_and_eval(ds, model_n, "Clean Val", "clean")
        
        # Experiment 2: Corrupted Validation (Using all 15 corruptions)
        train_and_eval(ds, model_n, "Corrupted Val All", "corrupted_all")
        
        # Experiment 3: Optimized Corruption Amount
        train_and_eval(ds, model_n, "Corrupted Val Optimized", "corrupted_optimized")

print("Training complete. Results saved in results/experiment_accuracies.csv")

In [ ]:
import torch  # Main PyTorch library for tensor operations
import torch.nn as nn  # Neural network modules (Linear, ReLU, etc.)
import torch.optim as optim  # Optimization algorithms (Adam, SGD)
from torch.utils.data import DataLoader, Subset, Dataset  # Data handling utilities
from torchvision import datasets, transforms  # Vision datasets and image processing
from imagecorruptions import corrupt  # Library for synthetic image corruptions 
import numpy as np  # Numerical computing library
from sklearn.model_selection import train_test_split  # Utility for consistent data splitting

# ==========================================
# 1. MODEL ARCHITECTURE (MLP)
# ==========================================
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, num_classes=10):
        super(SimpleMLP, self).__init__()  # Initialize the parent nn.Module
        self.flatten = nn.Flatten()  # Layer to convert 2D/3D images into a 1D feature vector
        # Defining the sequential layers as noted in your architectural analysis [cite: 378]
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 1024),  # First hidden layer (Input -> 1024)
            nn.BatchNorm1d(1024),        # Batch Normalization for training stability
            nn.ReLU(),                   # Activation function
            nn.Linear(1024, 512),        # Second hidden layer (1024 -> 512)
            nn.BatchNorm1d(512),         # Batch Normalization
            nn.ReLU(),                   # Activation function
            nn.Linear(512, 256),         # Third hidden layer (512 -> 256)
            nn.BatchNorm1d(256),         # Batch Normalization
            nn.ReLU(),                   # Activation function
            nn.Linear(256, num_classes)  # Output layer (256 -> number of classes)
        )

    def forward(self, x, apply_perturb=False):
        x = self.flatten(x)  # Flatten the input image to a 1D vector
        # Manually iterating through layers to allow middle-layer perturbation 
        for i, layer in enumerate(self.layers):
            x = layer(x)  # Pass the data through the current layer
            # Inject Gaussian noise N(0, 0.5^2) into the middle manifold for Exp 4 [cite: 244, 245]
            if i == 5 and apply_perturb and self.training:
                x = x + torch.randn_like(x) * 0.5  # Add stochastic noise to activations
        return x  # Return the final logits

# ==========================================
# 2. DATASET & CORRUPTION WRAPPER
# ==========================================
class CorruptedDataset(Dataset):
    def __init__(self, base_dataset, corruption_name=None, severity=2):
        self.base_dataset = base_dataset  # Store the original clean dataset
        self.corruption_name = corruption_name  # Specific corruption to apply 
        self.severity = severity  # Intensity of the corruption (set to 2 in report) 

    def __getitem__(self, index):
        img, target = self.base_dataset[index]  # Retrieve clean image and label
        if self.corruption_name:
            # Convert PyTorch tensor (C, H, W) to Numpy (H, W, C) for the library
            img_np = (img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
            # Apply synthetic corruption (e.g., Gaussian noise, blur) 
            corrupted_img = corrupt(img_np, corruption_name=self.corruption_name, severity=self.severity)
            # Convert back to a normalized PyTorch tensor
            img = transforms.ToTensor()(corrupted_img)
        return img, target  # Return the processed image and its original label

    def __len__(self):
        return len(self.base_dataset)  # Return the total number of samples

# ==========================================
# 3. TRAINING AND EVALUATION LOGIC
# ==========================================
def train_model(model, train_loader, val_loader, epochs=10, perturb=False):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)  # Optimizer for parameter updates [cite: 237]
    criterion = nn.CrossEntropyLoss()  # Loss function for classification 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Set hardware accelerator
    model.to(device)  # Move model weights to GPU/CPU

    for epoch in range(epochs):  # Loop through the dataset multiple times
        model.train()  # Set model to training mode (enables BatchNorm and Perturbation)
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)  # Move batch to device
            optimizer.zero_grad()  # Clear previous gradients
            outputs = model(images, apply_perturb=perturb)  # Forward pass with optional noise 
            loss = criterion(outputs, labels)  # Calculate the loss 
            loss.backward()  # Backpropagate gradients
            optimizer.step()  # Update model weights
        
        # Validation evaluation at the end of each epoch
        model.eval()  # Set to evaluation mode (disables BatchNorm updates and Noise)
        correct = 0  # Counter for accurate predictions
        with torch.no_grad():  # Disable gradient tracking to save memory
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)  # Standard forward pass
                correct += (outputs.argmax(1) == labels).sum().item()  # Count correct labels
        # Print validation accuracy to track progress
        print(f"Epoch {epoch+1}: Val Acc {100 * correct / len(val_loader.dataset):.2f}%")

# ==========================================
# 4. EXECUTION FLOW (Example: CIFAR-10)
# ==========================================
def run_experiment(exp_type="exp1"):
    # Standard transformation to convert PIL images to Tensors
    transform = transforms.Compose([transforms.ToTensor()])
    # Load CIFAR-10 training and test sets [cite: 369]
    full_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    test_data = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

    # Isolated 20% of training data for a fixed validation set 
    train_idx, val_idx = train_test_split(list(range(len(full_train))), test_size=0.2, random_state=42)
    train_subset = Subset(full_train, train_idx)  # 80% Training
    val_subset = Subset(full_train, val_idx)      # 20% Validation

    # Define validation set behavior based on the experiment number [cite: 241-244]
    if exp_type == "exp1":   # Experiment 1: Clean Validation
        val_set = val_subset
    elif exp_type == "exp2": # Experiment 2: Corrupted Validation 
        val_set = CorruptedDataset(val_subset, corruption_name='gaussian_noise', severity=2)
    elif exp_type == "exp3": # Experiment 3: Optimized Validation (Selective Corruptions) [cite: 243]
        val_set = CorruptedDataset(val_subset, corruption_name='shot_noise', severity=2)
    else:                    # Experiment 4: Feature Perturbation (Clean Val) 
        val_set = val_subset

    # Create DataLoaders to handle batching and shuffling
    train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=64)

    # Initialize MLP: CIFAR-10 images are 3x32x32, which flattens to 3072 features [cite: 369]
    model = SimpleMLP(input_dim=3072, num_classes=10)
    
    # Train the model with the perturbation flag active only for Exp 4
    perturb_flag = True if exp_type == "exp4" else False
    train_model(model, train_loader, val_loader, epochs=5, perturb=perturb_flag)

    return model  # Return the trained model for further testing